# Example: Empirical Transfer Function Estimate (ETFE)

This example demonstrates `sid.freq_etfe`, which estimates the frequency
response as the ratio of output and input DFTs. It provides maximum
frequency resolution but high variance. Optional smoothing reduces
variance at the cost of resolution.

Translated from `exampleETFE.m`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
import sid

%matplotlib inline

## Generate test data

True system: $G(z) = 1 / (1 - 0.8\,z^{-1})$ (first-order, pole at 0.8)

In [ ]:
rng = np.random.default_rng(1)

N = 1024
u = rng.standard_normal(N)
y_clean = lfilter([1], [1, -0.8], u)
y = y_clean + 0.3 * rng.standard_normal(N)

## Basic ETFE (no smoothing)

The raw ETFE has maximum resolution but is very noisy.
Note: `response_std` is NaN because ETFE has no asymptotic uncertainty formula.

In [ ]:
result = sid.freq_etfe(y, u)

sid.bode_plot(result)
plt.suptitle('ETFE - Raw (No Smoothing)', y=1.02)
plt.tight_layout()
plt.show()

## Effect of smoothing

Smoothing averages nearby frequency bins using a window of odd length S.
Larger S = smoother estimate but coarser resolution.

In [ ]:
r1  = sid.freq_etfe(y, u, smoothing=1)    # no smoothing (raw)
r11 = sid.freq_etfe(y, u, smoothing=11)   # moderate smoothing
r21 = sid.freq_etfe(y, u, smoothing=21)   # heavy smoothing

w = r1.frequency
G_true = 1.0 / (1.0 - 0.8 * np.exp(-1j * w))

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(w, 20 * np.log10(np.abs(r1.response)),  color='0.7', label='S = 1 (raw)')
ax.semilogx(w, 20 * np.log10(np.abs(r11.response)), 'b', label='S = 11')
ax.semilogx(w, 20 * np.log10(np.abs(r21.response)), 'r', label='S = 21')
ax.semilogx(w, 20 * np.log10(np.abs(G_true)), 'k--', lw=1.5, label='True')
ax.set_xlabel('Frequency (rad/sample)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('ETFE Smoothing Comparison')
ax.legend(loc='lower left')
ax.grid(True)
plt.tight_layout()
plt.show()

## Known FIR system: pure delay

For $y(t) = u(t-1)$, the true transfer function is $G(z) = z^{-1}$.
ETFE recovers this exactly when there is no noise.

In [ ]:
rng2 = np.random.default_rng(2)

N_fir = 1024
u_fir = rng2.standard_normal(N_fir)
y_fir = np.concatenate(([0.0], u_fir[:-1]))   # one-sample delay

result_fir = sid.freq_etfe(y_fir, u_fir)
w_fir = result_fir.frequency

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))

ax1.plot(w_fir, np.abs(result_fir.response), 'b')
ax1.set_ylabel('|G|')
ax1.set_title('ETFE of Pure Delay: |G| should be 1')
ax1.grid(True)

ax2.plot(w_fir, np.angle(result_fir.response), 'b', label='ETFE')
ax2.plot(w_fir, -w_fir, 'k--', label=r'True phase = $-\omega$')
ax2.set_ylabel('Phase (rad)')
ax2.set_xlabel('Frequency (rad/sample)')
ax2.legend(loc='lower left')
ax2.grid(True)

plt.tight_layout()
plt.show()

## Time-series mode: periodogram

With no input signal, `sid.freq_etfe` computes the periodogram of the output.

In [ ]:
rng3 = np.random.default_rng(3)

y_ts = lfilter([1], [1, -0.8], rng3.standard_normal(500))
result_ts = sid.freq_etfe(y_ts, None)

sid.spectrum_plot(result_ts)
plt.title('Periodogram of AR(1) Process')
plt.tight_layout()
plt.show()

## Custom frequency grid and Hz display

Use a logarithmic frequency grid and plot in Hz.

In [ ]:
rng4 = np.random.default_rng(4)

Ts = 0.001                                                  # 1 kHz sampling
w_log = np.logspace(np.log10(0.05), np.log10(np.pi), 200)  # log-spaced in rad/sample

N_hz = 2048
u_hz = rng4.standard_normal(N_hz)
y_hz = lfilter([1], [1, -0.9], u_hz) + 0.1 * rng4.standard_normal(N_hz)

result_hz = sid.freq_etfe(y_hz, u_hz, smoothing=11,
                          frequencies=w_log, sample_time=Ts)

sid.bode_plot(result_hz, frequency_unit='Hz')
plt.suptitle('ETFE with Log Frequency Grid (Hz)', y=1.02)
plt.tight_layout()
plt.show()